In [ ]:
# Change the branch to yours
# And choose an L4 GPU in Runtime > Change runtime type
BRANCH = "test-khalit-clean"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"


In [ ]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub


In [ ]:
# SSH test
%%bash
set -e
ssh -T git@github.com || true


In [ ]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR


In [ ]:
# Clone and checkout the branch indicated on the top
# After running, you should be able to see the repo in the Files in the left menu bar
%%bash
set -e

if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current
git log --oneline -1


In [ ]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"
echo -e "\n=== CPU Info ==="
lscpu | head -n 20
echo -e "\n=== Memory Info ==="
free -h


In [ ]:
# Install
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python vllm || true
python -m pip install -e python
python -m pip install -q pandas numpy requests tqdm


In [ ]:
# You should see the local SGLang package
import sys
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")
import sglang, torch
from importlib.metadata import version
print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
# Stop the SGLang server when needed
!pkill -f "sglang.launch_server" || true


In [ ]:
# Verify SGLang is gone
!ps aux | grep "sglang.launch_server" | grep -v grep


In [ ]:
# Combined cleanup before any new server launch
%%bash
pkill -f "sglang.launch_server" || true
sleep 2
ps aux | grep "sglang.launch_server" | grep -v grep || true


In [ ]:
# Launch the SGLang server with custom radix-cache-impl + custom cache-aware scheduling (HiCache off)
%%bash
cd /content/sglang
nohup python -m sglang.launch_server   --model-path Qwen/Qwen2.5-1.5B-Instruct   --port 30000   --radix-cache-impl custom   --cache-aware-scheduling custom   > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log


In [ ]:
# Check SGLang readiness
%%bash
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 100 /content/sglang_server.log


In [ ]:
# Run multi-turn chat benchmark against SGLang once
# Warmup, flush cache, then run the real benchmark
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/sglang_single.jsonl results/sglang_single_raw.json
python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file /tmp/sglang_warmup.jsonl > /tmp/bench_sglang_warmup.log
curl -s -X POST http://127.0.0.1:30000/flush_cache
sleep 2
python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file results/sglang_single.jsonl --raw-result-file results/sglang_single_raw.json --run-id single


In [ ]:
# Inspect the saved single-run SGLang result
%%bash
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
tail -n 1 results/sglang_single.jsonl


In [ ]:
# Run multi-turn chat benchmark against SGLang five times at parallel=1
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/sglang_p1_5runs.jsonl results/sglang_p1_run_*.json
python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file /tmp/sglang_warmup_loop.jsonl > /tmp/bench_sglang_warmup_loop.log
curl -s -X POST http://127.0.0.1:30000/flush_cache
sleep 2
for run_id in 1 2 3 4 5; do
  if [ "$run_id" != "1" ]; then
    curl -s -X POST http://127.0.0.1:30000/flush_cache
    sleep 2
  fi
  python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 1 --result-file results/sglang_p1_5runs.jsonl --raw-result-file results/sglang_p1_run_${run_id}.json --run-id "$run_id"
done


In [ ]:
# Run multi-turn chat benchmark against SGLang five times at parallel=16
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/multi_turn_chat_short
mkdir -p results
rm -f results/sglang_p16_5runs.jsonl results/sglang_p16_run_*.json
python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 16 --result-file /tmp/sglang_warmup_p16.jsonl > /tmp/bench_sglang_warmup_p16.log
curl -s -X POST http://127.0.0.1:30000/flush_cache
sleep 2
for run_id in 1 2 3 4 5; do
  if [ "$run_id" != "1" ]; then
    curl -s -X POST http://127.0.0.1:30000/flush_cache
    sleep 2
  fi
  python bench_multiturn_serving.py --backend sglang --tokenizer Qwen/Qwen2.5-1.5B-Instruct --parallel 16 --result-file results/sglang_p16_5runs.jsonl --raw-result-file results/sglang_p16_run_${run_id}.json --run-id "$run_id"
done


In [ ]:
import json
from pathlib import Path

import pandas as pd

def summarize(prefix):
    base = Path('/content/sglang/690AB/eval/experiment_1/multi_turn_chat_short/results')
    rows = []
    for run_id in range(1, 6):
        with (base / f'{prefix}_run_{run_id}.json').open() as f:
            run = json.load(f)
        row = {
            'run_id': run['run_id'],
            'mean_e2e_latency_ms': run['metrics']['mean_e2e_latency_ms'],
            'p90_e2e_latency_ms': run['extended_metrics']['p90_e2e_latency_ms'],
            'p95_e2e_latency_ms': run['extended_metrics'].get('p95_e2e_latency_ms'),
            'request_throughput': run['metrics']['request_throughput'],
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    metrics = []
    metrics.append({'metric': 'mean_e2e_latency_ms', 'mean': df['mean_e2e_latency_ms'].mean(), 'std': df['mean_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'p90_e2e_latency_ms', 'mean': df['p90_e2e_latency_ms'].mean(), 'std': df['p90_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'p95_e2e_latency_ms', 'mean': df['p95_e2e_latency_ms'].mean(), 'std': df['p95_e2e_latency_ms'].std(ddof=1)})
    metrics.append({'metric': 'request_throughput', 'mean': df['request_throughput'].mean(), 'std': df['request_throughput'].std(ddof=1)})
    return df, pd.DataFrame(metrics)

print('sglang_p1')
df, summary = summarize('sglang_p1')
display(df)
display(summary)

print('sglang_p16')
df, summary = summarize('sglang_p16')
display(df)
display(summary)
